# Mesh basic: planejamento com postes candidatos

Este exemplo atualiza os testes de `testes.ipynb`. Planejamos separadamente os enlaces Copel–Guaíra e Copel–Itália, buscando locais para repetidores entre as posições disponíveis. Os equipamentos propostos são definidos por perfis no projeto; um poste candidato só representa uma instalação na solução se for selecionado.

As células usam apenas chamadas ao SDK. O microserviço executa a busca e consulta o serviço de features.

## 1. Abrir o projeto

`scenario.toml` define os perfis e os parâmetros de busca em `[mesh]`. `nodes.csv` contém somente os equipamentos da rede: Copel precisa de conexão; Guaíra e Itália são acessos conectados. As colunas `connected` e `can_route` definem seus papéis.

`poles.csv` contém a infraestrutura disponível: `pole_id`, coordenadas, altura de instalação e `device_profile`, que descreve o rádio proposto caso o poste seja escolhido. A referência está em `[mesh.poles]` no TOML. Os postes não entram no cenário original como equipamentos que precisam de conexão. O SDK lê esse arquivo somente ao iniciar o planejamento do enlace.

A busca recebe interfaces terminais e filtra os candidatos por tecnologia, frequência, interface habilitada e capacidade de encaminhamento. Torre usa outro perfil de rádio da mesma tecnologia. Deneka tem duas interfaces e participa pela interface mesh. Shopping possui somente Wi-SUN e é excluído desta busca mesh. Os caminhos relativos consideram este notebook na pasta `notebooks` do workspace.

In [ ]:
from cisei_planning_sdk import PlanningClient, PlanningProject

client = PlanningClient("http://network-service:8080", workspace="..", timeout=1800)
project = PlanningProject("../projects/exemple_4_Mesh_basic")
scenario = client.open_project(project)

## 2. Planejar Copel–Guaíra

A busca começa pelo enlace direto e tenta inserir repetidores para melhorar os segmentos de maior custo. O destino já está marcado como `connected = true` nas entradas. A solução não altera o arquivo de entrada nem instala equipamentos. O resultado e o cache de features são salvos pelo SDK.

In [ ]:
proposal_guaira = scenario.plan_mesh_link("copel:d0:i0", "guaira:d0:i0", geo_base_url="http://features:8080")

### Alternativas avaliadas

O mapa seguinte mostra os enlaces que a busca efetivamente avaliou, inclusive alternativas descartadas. Clique em um enlace para consultar o custo e os campos `evaluated`, `expanded` e `selected`. `expanded` indica que o segmento foi explorado para procurar repetidores; `selected` indica participação no caminho final. Os candidatos compatíveis que não tiveram enlaces avaliados também permanecem visíveis.

In [ ]:
candidates_guaira = scenario.show_mesh_candidates()

### Caminho selecionado

A solução mostra somente os equipamentos e enlaces escolhidos. Os candidatos opcionais que não foram utilizados não são classificados como dispositivos sem serviço.

In [ ]:
view_guaira = scenario.show_mesh_link()

## 3. Planejar Copel–Itália

Este é um segundo estudo independente, usando o mesmo conjunto de candidatos e o cache do projeto. A altura de Itália é 30 m; a posição Torre oferece 100 m. A seleção depende da geometria, do relevo e do custo dos enlaces avaliados, não apenas da proximidade.

In [ ]:
proposal_italia = scenario.plan_mesh_link("copel:d0:i0", "italia:d0:i0", geo_base_url="http://features:8080")

### Alternativas avaliadas

O mapa seguinte mostra os enlaces que a busca efetivamente avaliou, inclusive alternativas descartadas. Clique em um enlace para consultar o custo e os campos `evaluated`, `expanded` e `selected`. `expanded` indica que o segmento foi explorado para procurar repetidores; `selected` indica participação no caminho final. Os candidatos compatíveis que não tiveram enlaces avaliados também permanecem visíveis.

In [ ]:
candidates_italia = scenario.show_mesh_candidates()

### Caminho selecionado

A solução mostra somente os equipamentos e enlaces escolhidos. Os candidatos opcionais que não foram utilizados não são classificados como dispositivos sem serviço.

In [ ]:
view_italia = scenario.show_mesh_link()

## 4. Interpretar os resultados

Os mapas permitem inspecionar os enlaces e os nós propostos. Os grafos de alternativas e as soluções permanecem no notebook, e cada par de extremos tem seus HTMLs próprios no projeto. `result.json` contém a última proposta e `features.json` contém as features brutas reutilizáveis.

A métrica deste exemplo é um custo de obstrução do relevo, com valor mínimo 1. A busca usa o maior custo entre os dois sentidos do enlace; o limite por segmento é 2 e podem ser inseridos até cinco repetidores. São parâmetros demonstrativos, não um modelo de RSSI calibrado. Os arquivos originais `nodes.csv` e `poles.csv` são preservados. Somente os repetidores selecionados aparecem como equipamentos na solução. O resultado inclui `mesh_candidates`, com o grafo avaliado e os custos dos enlaces. `mesh_search.excluded_candidates` registra os candidatos excluídos. O campo `mesh_search.meets_segment_threshold` registra se o caminho encontrado atende ao limite. Uma proposta pode continuar com segmentos acima dele quando não houver melhoria suficiente entre os candidatos disponíveis.

Para avaliar desempenho de uma tecnologia real, configure sua métrica e os perfis correspondentes. Estes dois estudos demonstram planejamento de enlaces com infraestrutura disponível; a incorporação das propostas a uma rede instalada é uma decisão posterior do engenheiro.